In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy.stats import chi2_contingency, pearsonr, spearmanr
from sklearn.feature_selection import mutual_info_classif, chi2, f_classif
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split

# delete (...)
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None) 


In [ ]:
df = pd.read_csv('group_26_train.csv')

In [ ]:
#number of samples and features
print("Dataset Shape:", df.shape)

df.head(5)

In [ ]:
# type of features!
print("List of all features:")
print(df.columns.tolist())

In [ ]:
print("\nDetailed feature information:")
print(df.dtypes)

In [ ]:
dtype_counts = df.dtypes.value_counts()
print(dtype_counts)

In [ ]:
numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns if col != 'y']
categorical_cols = df.select_dtypes(include=["str"]).columns.tolist()
bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print(numeric_cols)

print("\nCategorical columns:", len(categorical_cols))
print(categorical_cols)

print("\nBoolean columns:", len(bool_cols))
print(bool_cols)

In [ ]:
for feature in df.columns:
    # Check if feature is boolean, object (string), or categorical integer
    if df[feature].dtype in ['bool', 'str']:
        print(f"\n{feature}:")
        print(df[feature].value_counts())
        print("/////////")
    else:
        # For float/int, just skip or use describe
        pass


In [ ]:
df['f2'] = pd.to_datetime(df['f2'], format='mixed', errors='coerce')

df['hour'] = df['f2'].dt.hour
df['dayofweek'] = df['f2'].dt.dayofweek
df['month'] = df['f2'].dt.month
df['year'] = df['f2'].dt.year

df['dayofweek'] = df['dayofweek'].astype('str')
df['month'] = df['month'].astype('str')


display(df[['f2', 'year', 'month', 'dayofweek', 'hour']].head())

df = df.drop(columns=['f2'])



In [ ]:
df.head(5)

In [ ]:
day_night_cols = ['f31','f32', 'f33', 'f34']

for col in day_night_cols:
    df[col] = df[col].map({'Day': False, 'Night': True}).astype(bool)


In [ ]:
df.head(5)

In [ ]:
numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns if col != 'y']
categorical_cols = df.select_dtypes(include=["str"]).columns.tolist()
bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print(numeric_cols)

print("\nCategorical columns:", len(categorical_cols))
print(categorical_cols)

print("\nBoolean columns:", len(bool_cols))
print(bool_cols)

In [ ]:
#important y
target_counts = df['y'].value_counts().sort_index()
target_percent = df['y'].value_counts(normalize=True).sort_index() * 100

display(pd.DataFrame({
    'count': target_counts,
    'percent': target_percent
}))

fig, axes = plt.subplots(1,2, figsize=(14,5))

sns.countplot(data=df, x='y', ax=axes[0], palette='Set2')
axes[0].set_title('Target Distribution')
axes[0].set_xlabel('y')
axes[0].set_ylabel('Count')

axes[1].pie(
    target_counts.values,
    labels=target_counts.index,
    startangle=90,
    colors=sns.color_palette('Set2')
)

axes[1].set_title('Target Class Share')

plt.tight_layout()
plt.show()


In [ ]:
if 'numeric_cols' in locals() and 'y' in df.columns:
    # Data cleaning for MI
    cols_to_use = ['y'] + numeric_cols
    df_mi = df[cols_to_use].dropna()

    if df_mi['y'].nunique() >= 2:
        X_clean = df_mi[numeric_cols]
        y_clean = df_mi['y'].astype(int)

        mi_scores = mutual_info_classif(X_clean, y_clean, random_state=42)
        mi_series = pd.Series(mi_scores, index=numeric_cols).sort_values(ascending=False)

        print("Mutual Information Scores (Numerical vs y):")
        print(mi_series)

        plt.figure(figsize=(10, max(6, len(numeric_cols) * 0.4)))
        sns.barplot(x=mi_series.values, y=mi_series.index, palette='viridis')
        plt.title('Mutual Information: Numerical Features vs y')
        plt.xlabel('MI Score')
        plt.ylabel('Features')
        plt.tight_layout()
        plt.show()
    else:
        print("Insufficient data: Target 'y' has less than 2 unique classes after dropping NaNs.")
else:
    print("Missing 'numeric_cols' or target 'y' in DataFrame.")


In [ ]:
top_6_features = ['year','f6', 'f4', 'f7', 'f5', 'f3']

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(12, 12))
axes = axes.flatten()  # flatten to 1D array

for i, feature in enumerate(top_6_features):
    ax = axes[i]
    sns.histplot(df[feature].dropna(), kde=True, bins=30, color='skyblue', ax=ax)
    ax.set_title(f'Distribution of {feature}')
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')

# Remove the empty subplot if we have 5 features and 6 subplots (3x2)
if len(top_6_features) < len(axes):
    for j in range(len(top_6_features), len(axes)):
        fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
if 'categorical_cols' in locals() and 'y' in df.columns:
    df_clean = df[['y'] + categorical_cols].dropna(subset=['y'])
    
    if df_clean['y'].nunique() >= 2:
        df_encoded = pd.DataFrame()
        for col in categorical_cols:
            df_encoded[col] = df_clean[col].astype('category').cat.codes

        mi_scores = mutual_info_classif(df_encoded, df_clean['y'].astype(int), random_state=42)
        mi_series = pd.Series(mi_scores, index=categorical_cols).sort_values(ascending=False)

        print("Mutual Information Scores (Categorical vs y):")
        print(mi_series)

        plt.figure(figsize=(10, max(6, len(categorical_cols) * 0.4)))
        sns.barplot(x=mi_series.values, y=mi_series.index, palette='viridis')
        plt.title('Mutual Information: Categorical Features vs y')
        plt.xlabel('MI Score')
        plt.ylabel('Features')
        plt.tight_layout()
        plt.show()
    else:
        print("Target 'y' must have at least 2 classes after removing missing values.")
else:
    print("Missing 'categorical_cols' or target 'y' in DataFrame.")


In [ ]:
top_6_cat = ['f12', 'f8', 'f1', 'f9', 'f13', 'f10']

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(16, 18))
axes = axes.flatten()

for i, feature in enumerate(top_6_cat):
    counts = df[feature].value_counts().head(12)
    
    sns.barplot(x=counts.index, y=counts.values, palette='viridis', ax=axes[i])
    axes[i].set_title(f'{feature} (MI: {mi_series[feature]:.4f})')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45)  # Rotate labels for readability

plt.tight_layout()
plt.show()

In [ ]:

if 'bool_cols' in locals() and bool_cols and 'y' in df.columns:
    valid_bool_cols = [c for c in bool_cols if c in df.columns]

    df_mi = df[valid_bool_cols + ['y']].dropna()
    if df_mi['y'].nunique() >= 2 and valid_bool_cols:
        mi_scores = mutual_info_classif(df_mi[valid_bool_cols], df_mi['y'].astype(int), random_state=42)
        mi_series = pd.Series(mi_scores, index=valid_bool_cols).sort_values(ascending=False)

        print("Mutual Information Scores (Boolean Features vs y):")
        print(mi_series)

        plt.figure(figsize=(10, max(6, len(valid_bool_cols) * 0.4)))
        sns.barplot(x=mi_series.values, y=mi_series.index, palette='viridis')
        plt.title('Mutual Information: Boolean Features vs y')
        plt.xlabel('MI Score')
        plt.ylabel('Boolean Feature')
        plt.tight_layout()
        plt.show()
    else:
        print("Insufficient valid data for Mutual Information analysis.")
elif 'y' not in df.columns:
    print("Missing target column 'y'.")
else:
    print("No boolean columns found.")


In [ ]:
# Check distribution of boolean features
bool_cols = ['f22', 'f23', 'f24', 'f25', 'f26', 'f27', 'f28', 'f29', 'f30', 'f31', 'f32', 'f33', 'f34']

for col in bool_cols:
    if col in df.columns:
        print(f"{col}: True={df[col].sum()}, False={len(df) - df[col].sum()}, %True={df[col].mean()*100:.1f}%")

In [ ]:
binary_cols = [col for col in df.columns if df[col].dropna().isin([True, False]).all()]
print('Binary columns:', binary_cols)

binary_summary = pd.DataFrame({
    'true_count': df[binary_cols].sum(numeric_only=True),
    'false_count': df[binary_cols].shape[0] - df[binary_cols].sum(numeric_only=True)
    
})
display(binary_summary)

binary_summary['true_ratio'] = binary_summary['true_count'] / (binary_summary['true_count'] + binary_summary['false_count'])
plt.figure(figsize=(14, 6))
sns.barplot(x=binary_summary.index, y=binary_summary['true_ratio'], palette='Set3')
plt.xticks(rotation=90)
plt.title('True Ratio of Binary Features')
plt.ylabel('True Ratio')
plt.xlabel('Binary Features')
plt.tight_layout()
plt.show()


In [ ]:
if numeric_cols:
    # Pearson Correlation Heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(df[numeric_cols].corr(method='pearson'), annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
    plt.title('Pearson Correlation Heatmap')
    plt.show()

    # Spearman Correl ation Heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(df[numeric_cols].corr(method='spearman'), annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
    plt.title('Spearman Correlation Heatmap')
    plt.show()
else:
    print("No numerical columns found for correlation analysis.")


In [ ]:
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    if confusion_matrix.empty or confusion_matrix.to_numpy().sum() == 0:
        return np.nan
    
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.to_numpy().sum()
    r, k = confusion_matrix.shape
    
    if min(r - 1, k - 1) <= 0:
        return 0.0
    return np.sqrt((chi2 / n) / min(r - 1, k - 1))

sample_df = df.sample(min(5000, len(df)), random_state=42)
cat_cols = [c for c in sample_df.select_dtypes(include=['object', 'category']).columns if c != 'f2']

print(f"Categorical columns used: {cat_cols}")

cramers_matrix = pd.DataFrame(np.eye(len(cat_cols)), index=cat_cols, columns=cat_cols)

for i, col1 in enumerate(cat_cols):
    for j, col2 in enumerate(cat_cols):
        if i < j:
            val = cramers_v(sample_df[col1], sample_df[col2])
            cramers_matrix.loc[col1, col2] = val
            cramers_matrix.loc[col2, col1] = val

display(cramers_matrix)

plt.figure(figsize=(10, 8))
sns.heatmap(cramers_matrix, annot=True, cmap='Blues', vmin=0, vmax=1, fmt=".2f")
plt.title("Cramer's V Heatmap (Categorical Features)")
plt.tight_layout()
plt.show()
